# Day 06 — NumPy ve Vektörel Hesaplama
## ndarray Yapısı, Vektörizasyon, Yayınlama (Broadcasting) ve Bellek Yerleşimi

> **Aşama:** Faz 1 — Problem, Veri ve Geliştirme Temelleri (Day 01–08)
> **Resmi Staj Defteri Konusu:** NumPy ve Vektörel Hesaplama (Yaprak 11 & 12)

### 1. Problem
Tekstil üretiminde binlerce sensör kaydı veya milyonlarca piksel içeren görüntüler üzerinde standart Python `for` döngüleriyle matematiksel işlemler yapmak aşırı CPU yükü ve kabul edilemez gecikmelere yol açar. Sayısal hesaplamaların C seviyesinde SIMD (Single Instruction Multiple Data) vektörel optimizasyonla yürütülmesi şarttır.

### 2. Why the Problem Matters
NumPy `ndarray` veri yapısı bitişik (contiguous) bellek bloklarında çalışarak CPU önbellek (L1/L2/L3) kaçırma oranını minimize eder ve döngüsüz hesaplama ile 20x–100x hızlanma sağlar.

### 3. Engineering Concepts
- **Bitişik Bellek (Memory Contiguity & Strides)**: C-sıralı (row-major) bellek dizilimi.
- **Yayınlama (Broadcasting)**: Farklı boyutlardaki dizilerin bellek kopyalaması yapmadan matematiksel işleme girmesi.
- **Vektörel Ölçekleme**: Min-Max ve Z-Score standardizasyonunun matris seviyesinde hesaplanması.

In [ ]:
# 4. Library / API Investigation
import numpy as np
print(f"NumPy Version: {np.__version__}")

In [ ]:
# 5. Minimal Implementation
import numpy as np

class MinMaxScaler:
    def __init__(self):
        self.min_ = None
        self.max_ = None

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        denom = np.where((self.max_ - self.min_) == 0, 1.0, (self.max_ - self.min_))
        return (X - self.min_) / denom

class StandardScaler:
    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        denom = np.where(self.std_ == 0, 1.0, self.std_)
        return (X - self.mean_) / denom

X = np.array([
    [70.0, 14.0, 800.0],
    [85.0, 15.2, 830.0],
    [65.0, 13.8, 790.0],
    [90.0, 16.0, 860.0]
], dtype=np.float64)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Orijinal Veri Matrisi:\n", X)
print("\nZ-Score Standartlaştırılmış Matris:\n", np.round(X_scaled, 4))


In [ ]:
# 6. Experiment: Loop vs Vectorized Speedup
import time
large_arr = np.random.rand(500000, 3)

# Python döngüsü
t0 = time.perf_counter()
loop_res = []
for row in large_arr[:10000]:  # Yalnızca 10.000 örnek
    loop_res.append(row[0] * 0.299 + row[1] * 0.587 + row[2] * 0.114)
t_loop = (time.perf_counter() - t0) * 50 # 500.000 ölçeğine projeksiyon

# NumPy vektörel işlem
t0 = time.perf_counter()
weights = np.array([0.299, 0.587, 0.114])
vec_res = large_arr @ weights
t_vec = time.perf_counter() - t0

print(f"Döngü Süresi (tahmini): {t_loop:.4f}s | Vektörel Süre: {t_vec:.4f}s")
print(f"Hızlanma Faktörü: {t_loop / t_vec:.1f}x")

In [ ]:
# 7. Visualization
import matplotlib.pyplot as plt

methods = ["Python For Döngüsü", "NumPy Vektörel"]
times = [t_loop, t_vec]

plt.figure(figsize=(5, 3.5))
plt.bar(methods, times, color=["#d95f02", "#1b9e77"], width=0.4)
plt.yscale("log")
plt.ylabel("Süre (Saniye - Logaritmik Ölçek)")
plt.title("NumPy Vektörizasyon Hesaplama Hızı")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# 8. Validation
assert np.allclose(X_scaled.mean(axis=0), [0, 0, 0], atol=1e-7)
assert np.allclose(X_scaled.std(axis=0), [1, 1, 1], atol=1e-7)
print("Z-Score normalizasyonu ortalama 0 ve standart sapma 1 olarak doğrulandı.")

In [ ]:
# 9. Failure Cases: Boyut Uyuşmazlığı
try:
    large_arr @ np.array([1.0, 2.0])  # 3 boyutlu matrise 2 elemanlı çarpım
except ValueError as e:
    print("Beklenen boyut uyuşmazlığı hatası yakalandı:", e)

### 10. Conclusions
NumPy ile vektörel hesaplama mantığı incelenmiş, bitişik bellek ve SIMD optimizasyonunun CPU işlem hızını onlarca kat artırdığı deneysel olarak gösterilmiştir.